# Mobile VLM Production Issues — Papers, Identify, Solve

*Part 5 of 5 · Research-backed diagnosis and fixes for on-device OCR*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gaurav14cs17/Document-OCR-Pipeline/blob/main/colab/05_mobile_production_issues.ipynb)

Companion to [04 — Mobile Complete](04_ocr_pipeline_mobile_complete.ipynb).

This notebook is organized like a **literature review + engineering playbook**:

1. **Read the papers** — what researchers discovered about each bottleneck
2. **Identify** — detect the problem on your device (Part A)
3. **Solve** — apply the paper-backed fix (Part B)
4. **Verify** — scorecard before shipping (Part C)

```
Stage 0   Install + config
Stage 1   Shared utilities

── PART A: IDENTIFY ──
Stage 2   KV cache      (literature → detect → measure)
Stage 3   Power         (literature → detect → measure)
Stage 4   Quant loss    (literature → detect → measure)
Stage 5   RAM           (literature → detect → measure)

── PART B: SOLVE ──
Stage 6   KV cache fixes     (paper → method → code)
Stage 7   Power fixes        (paper → method → code)
Stage 8   Quant fixes        (paper → method → code)
Stage 9   RAM fixes          (paper → method → code)

── PART C: VERIFY ──
Stage 10  Scorecard + export
```

**Series:** [01 OCR](01_document_ocr_pipeline.ipynb) → [02 Quant](02_ocr_pipeline_quant.ipynb) → [03 Mobile](03_ocr_pipeline_mobile.ipynb) → [04 Complete](04_ocr_pipeline_mobile_complete.ipynb) → **05 Issues**


## Stage 0 — Install & config


In [ ]:
import os, re, sys, subprocess, math, json, time
from dataclasses import dataclass, field
from typing import Callable

# ── CONFIG — change these before you run ─────────────────

# ── Model & Task ──
MODEL_ID = "microsoft/Florence-2-base-ft"   # VLM used for OCR benchmarks
TASK = "detect"                              # detect | ocr

# ── Phone Budget Targets ──
TARGET_RAM_MB = 512                          # peak RAM budget (MB)
TARGET_DISK_MB = 150                         # on-device bundle size
TARGET_LATENCY_SEC = 8.0                     # max seconds per page
TARGET_MAH_PER_PAGE = 15.0                   # max battery drain per page
BATTERY_MAH = 3000.0                         # phone battery for % estimates

# ── Model Architecture (Florence-2-base rough) ──
MAX_NEW_TOKENS = 256                         # tokens generated in benchmarks
NUM_LAYERS = 6                               # decoder transformer layers L
HIDDEN_DIM = 1024                            # hidden size d
NUM_Q_HEADS = 16                             # query attention heads h
NUM_KV_HEADS = 16                            # KV heads h_kv (GQA: set < NUM_Q_HEADS)
KV_BYTES_PER_ELEM = 2                        # fp16=2, int8 KV=1

# ── Power Model ──
CHIP_TDP_W = 5.0                             # sustained SoC power (W)
EFFICIENCY_J_PER_GFLOP = 0.8                 # energy per GFLOP (calibrate on device)
PHONE_GFLOPS = 2.5                           # effective GFLOP/s on mid-range NPU
THERMAL_THROTTLE_SEC = 10.0                  # sustained inference before throttle

# ── Quant Precision ──
QUANT_BITS = 4                               # default quant bit-width
MIN_OCR_LINE_RECALL = 0.85                   # quality gate threshold
FP16_SENSITIVE_PCT = 15                        # top sensitive layers → fp16
ALWAYS_FP16_PATTERNS = ("lm_head", "embed", "visual_projection", "vision")

# ── Image ──
MAX_IMAGE_SIZE = 768                           # long-edge px on phone
PARAMS_M = 230.0                               # model params in millions


def ensure_transformers():
    """Pin transformers==4.49 for Florence-2 compatibility."""
    r = subprocess.run(
        [sys.executable, "-m", "pip", "show", "transformers"],
        capture_output=True, text=True,
    )
    if not re.search(r"^Version: 4\.49", r.stdout, re.M):
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q",
            "--force-reinstall", "transformers==4.49.0",
        ])

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "numpy>=1.26", "scipy>=1.12", "scikit-learn",
    "torch", "pillow", "matplotlib", "requests", "huggingface_hub"])
ensure_transformers()
print(f"Stage 0 done — {MODEL_ID}  RAM≤{TARGET_RAM_MB}MB  K≤{MAX_NEW_TOKENS}")

---
## Stage 1 — Shared utilities

Measurement helpers used in Stages 2–10. All from scratch (no profiler SDKs).


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")


# ═══════════════════════════════════════════════════════════════
# Shared data structures
# ═══════════════════════════════════════════════════════════════

@dataclass
class DetectionResult:
    """One row in an identification report."""
    method_id: int
    name: str
    measured: str
    threshold: str
    status: str          # "OK ✓" | "DETECTED ✗" | "manual"
    is_problem: bool = False


@dataclass
class FixResult:
    """One row in a solve/mitigation report."""
    name: str
    paper: str
    priority: str
    value: float           # MB, mAh, or peak MB depending on context
    unit: str
    savings_pct: float     # vs baseline


# ═══════════════════════════════════════════════════════════════
# Algorithm 1: KV cache memory (Pope et al. 2023)
# ═══════════════════════════════════════════════════════════════

@dataclass
class KVCacheConfig:
    """Architecture knobs that affect KV RAM."""
    num_layers: int          # L
    seq_len: int             # K — generated tokens
    hidden_dim: int          # d
    num_kv_heads: int        # h_kv
    bytes_per_elem: int = 2  # fp16=2, int8=1

    @property
    def head_dim(self) -> int:
        return self.hidden_dim // max(self.num_kv_heads, 1)  # d_h = d / h_kv

    @property
    def bytes_total(self) -> int:
        # KV = 2 tensors (K+V) × L × K × h_kv × d_h × elem_size
        return 2 * self.num_layers * self.seq_len * self.num_kv_heads * self.head_dim * self.bytes_per_elem

    @property
    def mb(self) -> float:
        return self.bytes_total / (1024 ** 2)

    def with_overrides(self, **kwargs) -> "KVCacheConfig":
        d = {**self.__dict__}
        d.update(kwargs)
        return KVCacheConfig(**d)


class KVCacheAnalyzer:
    """Identify KV cache problems — Pope et al. formula + token sweep."""

    def __init__(self, cfg: KVCacheConfig, ram_budget_mb: float):
        self.cfg = cfg
        self.kv_budget_mb = ram_budget_mb * 0.4  # KV should stay ≤ 40% of RAM

    def formula_mb(self) -> float:
        return self.cfg.mb

    def find_oom_token(self, token_list: list[int]) -> int | None:
        """First K in token_list where KV alone exceeds budget."""
        for k in token_list:
            if KVCacheConfig(self.cfg.num_layers, k, self.cfg.hidden_dim,
                             self.cfg.num_kv_heads, self.cfg.bytes_per_elem).mb > self.kv_budget_mb:
                return k
        return None

    def latency_slope_ms_per_token(self, k_values: list[int]) -> float:
        """Proxy: attention reads full KV each step → latency grows ~linearly."""
        if len(k_values) < 2:
            return 0.0
        mbs = [KVCacheConfig(self.cfg.num_layers, k, self.cfg.hidden_dim,
                             self.cfg.num_kv_heads, 2).mb for k in k_values]
        return (mbs[-1] - mbs[0]) / max(k_values[-1] - k_values[0], 1) * 100  # rough ms/MB

    def identify(self) -> tuple[list[DetectionResult], bool]:
        kv_mb = self.formula_mb()
        ratio = kv_mb / max(TARGET_RAM_MB, 1)
        crash_k = self.find_oom_token([32, 64, 128, 256, 512])
        oom_early = crash_k is not None and crash_k < 128

        rows = [
            DetectionResult(1, "Formula (Pope 2023)", f"{kv_mb:.1f} MB", f"≤ {self.kv_budget_mb:.0f} MB",
                            "DETECTED ✗" if kv_mb > self.kv_budget_mb else "OK ✓", kv_mb > self.kv_budget_mb),
            DetectionResult(2, "OOM crash", f"K≈{crash_k}" if crash_k else "none", "no crash < K=128",
                            "DETECTED ✗" if oom_early else "OK ✓", oom_early),
            DetectionResult(3, "Latency growth", "ms/token ↑ with K", "flat slope", "manual"),
            DetectionResult(4, "Memory profiler", "RSS slope > 0", "flat after vision", "manual"),
            DetectionResult(5, "KV/budget ratio", f"{ratio:.0%}", "≤ 40%",
                            "DETECTED ✗" if ratio > 0.4 else "OK ✓", ratio > 0.4),
            DetectionResult(6, "Token cap sweep", f"OOM @ K={crash_k}" if crash_k else "OK", "survive K≥128",
                            "DETECTED ✗" if oom_early else "OK ✓", oom_early),
        ]
        problem = any(r.is_problem for r in rows)
        return rows, problem

    def growth_curve(self, max_k: int, steps: int = 20) -> tuple[list[int], list[float]]:
        ks = list(range(0, max_k + 1, max(1, max_k // steps)))
        mbs = [KVCacheConfig(self.cfg.num_layers, k, self.cfg.hidden_dim,
                             self.cfg.num_kv_heads, 2).mb for k in ks]
        return ks, mbs


# ═══════════════════════════════════════════════════════════════
# Algorithm 2: Power / energy model (Orca + Leviathan)
# ═══════════════════════════════════════════════════════════════

@dataclass
class FLOPBreakdown:
    """FLOP estimate for one OCR page."""
    vision_gflops: float
    decode_gflops: float

    @property
    def total_gflops(self) -> float:
        return self.vision_gflops + self.decode_gflops


@dataclass
class PowerEstimate:
    """Energy model: E = η·FLOPs_eff, P = E/t, mAh = E/(V·3600)·1000."""
    flop: FLOPBreakdown
    seconds: float
    effective_gflops: float = 0.0   # after quant / speculative multipliers
    efficiency_j_per_gflop: float = EFFICIENCY_J_PER_GFLOP
    battery_voltage: float = 3.7

    def __post_init__(self):
        if self.effective_gflops <= 0:
            self.effective_gflops = self.flop.total_gflops

    @property
    def joules(self) -> float:
        return self.effective_gflops * self.efficiency_j_per_gflop

    @property
    def mah(self) -> float:
        return self.joules / self.battery_voltage * 1000 / 3600

    @property
    def watts_avg(self) -> float:
        return self.joules / max(self.seconds, 1e-6)

    @property
    def thermal_risk(self) -> str:
        if self.seconds > 10 or self.watts_avg > 4:
            return "HIGH"
        if self.seconds > 5 or self.watts_avg > 2.5:
            return "MEDIUM"
        return "LOW"


class PowerAnalyzer:
    """Identify power problems — FLOP-based energy + thermal proxy."""

    def estimate_flops(self, image_side: int, num_tokens: int,
                       params_m: float = PARAMS_M) -> FLOPBreakdown:
        # Vision: ~2× params, scales with (s/768)²  [Pope et al.]
        vision = 2.0 * params_m * (image_side / 768.0) ** 2
        # Decode: K steps × ~2× lang_params per step  [Orca]
        lang_frac = 0.6
        decode = num_tokens * 2.0 * params_m * lang_frac
        return FLOPBreakdown(vision, decode)

    def estimate_power(self, image_side: int, num_tokens: int,
                       quant: bool = False, speculative: bool = False,
                       gflops_per_sec: float = PHONE_GFLOPS) -> PowerEstimate:
        flop = self.estimate_flops(image_side, num_tokens)
        gflops = flop.total_gflops
        if quant:
            gflops *= 0.70   # int4 cuts DRAM traffic ~30%
        if speculative:
            gflops *= 0.60   # Leviathan 2023 — ~40% fewer full-model steps
        seconds = gflops / max(gflops_per_sec, 1e-6)
        return PowerEstimate(flop, seconds, effective_gflops=gflops)

    def identify(self, image_side: int, num_tokens: int) -> tuple[list[DetectionResult], bool, PowerEstimate]:
        est = self.estimate_power(image_side, num_tokens, quant=False)
        rows = [
            DetectionResult(1, "mAh (FLOP model)", f"{est.mah:.1f}", f"≤ {TARGET_MAH_PER_PAGE}",
                            "DETECTED ✗" if est.mah > TARGET_MAH_PER_PAGE else "OK ✓",
                            est.mah > TARGET_MAH_PER_PAGE),
            DetectionResult(2, "Wall time (Orca)", f"{est.seconds:.1f}s", f"≤ {TARGET_LATENCY_SEC}s",
                            "DETECTED ✗" if est.seconds > TARGET_LATENCY_SEC else "OK ✓",
                            est.seconds > TARGET_LATENCY_SEC),
            DetectionResult(3, "Thermal", est.thermal_risk, "LOW",
                            "DETECTED ✗" if est.thermal_risk == "HIGH" else "OK ✓",
                            est.thermal_risk == "HIGH"),
            DetectionResult(4, "Battery Historian", f"{est.watts_avg:.1f}W", "< 3W",
                            "DETECTED ✗" if est.watts_avg > 3 else "OK ✓", est.watts_avg > 3),
            DetectionResult(5, "CPU throttle", "freq drop", "< 30%", "manual"),
            DetectionResult(6, "Repeat slowdown", "2nd page", "< 20% slower", "manual"),
        ]
        problem = any(r.is_problem for r in rows)
        return rows, problem, est


# ═══════════════════════════════════════════════════════════════
# Algorithm 3: Quant precision (GPTQ/AWQ/SqueezeLLM metrics)
# ═══════════════════════════════════════════════════════════════

def qmax_for_bits(n_bits: int) -> int:
    return 2 ** (n_bits - 1) - 1


def symmetric_quantize_per_channel(W: torch.Tensor, n_bits: int = 4) -> tuple[torch.Tensor, torch.Tensor]:
    """Per-output-row symmetric quant — same primitive as notebook 02."""
    qmax = qmax_for_bits(n_bits)
    W = W.float()
    scales = W.abs().amax(dim=1, keepdim=True).clamp(min=1e-8) / qmax
    q = torch.round(W / scales).clamp(-qmax - 1, qmax)
    w_hat = q * scales
    return w_hat, scales.squeeze(1)


@dataclass
class LayerQuantProfile:
    name: str
    weight_mse: float
    num_params: int
    protected: bool
    sensitivity: float   # weight_mse × log(1 + params) — SqueezeLLM proxy


@dataclass
class QuantLossReport:
    layers: list[LayerQuantProfile]
    compression_ratio: float
    ocr_line_recall: float
    ocr_char_error_rate: float

    @property
    def weight_mse_mean(self) -> float:
        return sum(l.weight_mse for l in self.layers) / max(len(self.layers), 1)

    @property
    def weight_mse_max(self) -> float:
        return max((l.weight_mse for l in self.layers), default=0.0)

    @property
    def passes_quality_gate(self) -> bool:
        return self.ocr_line_recall >= MIN_OCR_LINE_RECALL


class QuantAnalyzer:
    """Identify quant precision loss — per-layer RTN MSE + OCR recall proxy."""

    def __init__(self, model: nn.Module, bits: int = 4):
        self.model = model
        self.bits = bits

    def profile_layers(self, max_layers: int = 40) -> list[LayerQuantProfile]:
        rows = []
        for name, mod in self.model.named_modules():
            if not isinstance(mod, nn.Linear):
                continue
            W = mod.weight.data
            w_hat, _ = symmetric_quantize_per_channel(W, self.bits)
            mse = (W.float() - w_hat).pow(2).mean().item()
            n = W.numel()
            protected = any(p in name.lower() for p in ALWAYS_FP16_PATTERNS)
            sens = mse * (1.0 + 0.1 * math.log1p(n))
            rows.append(LayerQuantProfile(name, mse, n, protected, sens))
            if len(rows) >= max_layers:
                break
        rows.sort(key=lambda r: r.sensitivity, reverse=True)
        return rows

    def identify(self, layers: list[LayerQuantProfile],
                 recall: float = 1.0) -> tuple[list[DetectionResult], bool]:
        max_mse = max((l.weight_mse for l in layers), default=0.0)
        mean_mse = sum(l.weight_mse for l in layers) / max(len(layers), 1)
        out_mse_proxy = mean_mse * 10.0

        rows = [
            DetectionResult(1, "Weight MSE (GPTQ)", f"max={max_mse:.2e}", "< 1e-3",
                            "DETECTED ✗" if max_mse > 1e-3 else "OK ✓", max_mse > 1e-3),
            DetectionResult(2, "Output MSE (AWQ)", f"{out_mse_proxy:.2e}", "< 1e-2",
                            "DETECTED ✗" if out_mse_proxy > 1e-2 else "OK ✓", out_mse_proxy > 1e-2),
            DetectionResult(3, "Line recall", f"{recall:.0%}", f"≥ {MIN_OCR_LINE_RECALL:.0%}",
                            "DETECTED ✗" if recall < MIN_OCR_LINE_RECALL else "OK ✓",
                            recall < MIN_OCR_LINE_RECALL),
            DetectionResult(4, "CER", "A/B in nb02", "< 5%", "manual"),
            DetectionResult(5, "Box IoU", "A/B in nb01", "≥ 0.85", "manual"),
            DetectionResult(6, "Visual inspection", "eyeball", "no regression", "manual"),
        ]
        problem = any(r.is_problem for r in rows)
        return rows, problem


# ═══════════════════════════════════════════════════════════════
# Algorithm 4: RAM peak (Pope 2023 + FlexGen tiering)
# ═══════════════════════════════════════════════════════════════

@dataclass
class RAMBreakdown:
    weights_mb: float
    image_bitmap_mb: float
    image_tensor_mb: float
    vision_activation_mb: float
    kv_cache_mb: float
    decode_activation_mb: float
    scratch_mb: float = 32.0

    @property
    def peak_mb(self) -> float:
        return (self.weights_mb + self.image_bitmap_mb + self.image_tensor_mb +
                self.vision_activation_mb + self.kv_cache_mb + self.decode_activation_mb + self.scratch_mb)

    def as_dict(self) -> dict[str, float]:
        return {
            "weights": self.weights_mb, "image": self.image_bitmap_mb + self.image_tensor_mb,
            "vision_act": self.vision_activation_mb, "KV": self.kv_cache_mb,
            "decode_act": self.decode_activation_mb, "scratch": self.scratch_mb,
        }

    def dominant(self) -> str:
        return max(self.as_dict(), key=self.as_dict().get)

    def pct(self, key: str) -> float:
        return 100.0 * self.as_dict()[key] / max(self.peak_mb, 1e-6)


class RAMAnalyzer:
    """Peak RAM estimator — Pope et al. component model."""

    def estimate(self, weights_mb: float, image_side: int, kv_cfg: KVCacheConfig,
                 vision_act_ratio: float = 0.15) -> RAMBreakdown:
        bitmap = image_side ** 2 * 4 / (1024 ** 2)       # RGBA camera buffer
        tensor = image_side ** 2 * 3 * 4 / (1024 ** 2)   # fp32 pixel_values
        vision_act = weights_mb * vision_act_ratio
        kv_mb = kv_cfg.mb
        decode_act = kv_cfg.hidden_dim * kv_cfg.seq_len * 4 / (1024 ** 2) * 0.5
        return RAMBreakdown(weights_mb, bitmap, tensor, vision_act, kv_mb, decode_act)

    def identify(self, peak_mb: float, breakdown: RAMBreakdown,
                 fp16_peak: float) -> tuple[list[DetectionResult], bool]:
        dom = breakdown.dominant()
        dom_pct = breakdown.pct(dom)
        camera_overlap = breakdown.image_bitmap_mb + breakdown.weights_mb

        rows = [
            DetectionResult(1, "Peak formula", f"{peak_mb:.0f} MB", f"≤ {TARGET_RAM_MB}",
                            "DETECTED ✗" if peak_mb > TARGET_RAM_MB else "OK ✓", peak_mb > TARGET_RAM_MB),
            DetectionResult(2, "Dominant term", f"{dom} ({dom_pct:.0f}%)", "< 50% one term",
                            "DETECTED ✗" if dom_pct > 50 else "OK ✓", dom_pct > 50),
            DetectionResult(3, "Memory profiler", "RSS", f"≤ {TARGET_RAM_MB}", "manual"),
            DetectionResult(4, "Low-mem kill", "onTrimMemory", "no kill", "manual"),
            DetectionResult(5, "fp16 gap", f"{fp16_peak:.0f}→{peak_mb:.0f} MB", "int4 fits",
                            "DETECTED ✗" if fp16_peak > TARGET_RAM_MB else "OK ✓", fp16_peak > TARGET_RAM_MB),
            DetectionResult(6, "Camera overlap", f"{camera_overlap:.0f} MB", f"< {TARGET_RAM_MB*0.8:.0f}",
                            "DETECTED ✗" if camera_overlap > TARGET_RAM_MB * 0.8 else "OK ✓",
                            camera_overlap > TARGET_RAM_MB * 0.8),
        ]
        problem = any(r.is_problem for r in rows)
        return rows, problem


# ═══════════════════════════════════════════════════════════════
# Reporting helpers
# ═══════════════════════════════════════════════════════════════

def print_detection_report(title: str, rows: list[DetectionResult], problem: bool, next_stage: str):
    print(f"{title}\n")
    print(f"  {'Method':<26} {'Measured':>16} {'Threshold':>16}  Result")
    print("  " + "-" * 68)
    for r in rows:
        print(f"  {r.name:<26} {r.measured:>16} {r.threshold:>16}  {r.status}")
    print("  " + "-" * 68)
    print(f"  PROBLEM: {'YES → ' + next_stage if problem else 'NO'}\n")


def print_fix_report(title: str, baseline: float, fixes: list[FixResult], unit: str):
    print(f"{title}\n  Baseline: {baseline:.2f} {unit}\n")
    print(f"  {'Fix':<30} {'Paper':<22} {'Value':>8}  {'Saved':>8}")
    print("  " + "-" * 72)
    for f in fixes:
        print(f"  {f.name:<30} {f.paper:<22} {f.value:7.2f}{f.unit}  {f.savings_pct:7.0%}")
    print()


@dataclass
class ProductionScorecard:
    """Part C verification — before/after against phone budgets."""
    kv_mb: float
    kv_budget: float
    mah: float
    mah_budget: float
    ram_mb: float
    ram_budget: float
    recall: float
    recall_min: float
    latency_s: float
    latency_budget: float
    label: str = ""

    def rows(self) -> list[tuple[str, str, str, bool]]:
        return [
            ("KV cache", f"{self.kv_mb:.1f}MB", f"≤{self.kv_budget:.0f}MB", self.kv_mb <= self.kv_budget),
            ("Power", f"{self.mah:.1f}mAh", f"≤{self.mah_budget:.0f}", self.mah <= self.mah_budget),
            ("RAM", f"{self.ram_mb:.0f}MB", f"≤{self.ram_budget:.0f}", self.ram_mb <= self.ram_budget),
            ("Quant recall", f"{self.recall:.0%}", f"≥{self.recall_min:.0%}", self.recall >= self.recall_min),
            ("Latency", f"{self.latency_s:.1f}s", f"≤{self.latency_budget:.0f}s", self.latency_s <= self.latency_budget),
        ]

    def print_card(self):
        print(f"\n{self.label}")
        for issue, measured, budget, ok in self.rows():
            print(f"  {issue:<14} {measured:>10}  budget {budget:>10}  {'✓' if ok else '✗'}")


print("Stage 1 done — analyzers + ProductionScorecard ready")

---
# PART A — IDENTIFY

---
## Stage 2 — Problem 1: KV cache

### Step 2.1 — Literature: why KV cache exists

Autoregressive transformers cache past **keys** and **values** so each new token does not recompute attention over the full prefix. Introduced implicitly in the original Transformer:

> **Attention Is All You Need** — Vaswani et al., NeurIPS 2017  
> [arxiv.org/abs/1706.03762](https://arxiv.org/abs/1706.03762)

Memory per layer grows as:

$$\text{KV}_\ell = 2 \cdot K \cdot h_{\text{kv}} \cdot d_h \cdot \text{sizeof(elem)}$$

For $L$ layers total KV $= L \times \text{KV}_\ell$ — **linear in sequence length $K$**.

### Step 2.2 — Key papers (KV cache research landscape)

| Paper | Authors | Year | Core idea | Mobile relevance |
|-------|---------|------|-----------|------------------|
| **Efficiently Scaling Transformer Inference** | Pope et al. | MLSys 2023 | Formal KV memory analysis; batching breaks memory | [arxiv.org/abs/2207.04910](https://arxiv.org/abs/2207.04910) |
| **Fast Transformer Decoding: One Write-Head is All You Need** | Shazeer | 2019 | **MQA** — one KV head for all Q heads | $N_h \times$ KV savings |
| **GQA: Training Generalized Multi-Query Attention** | Ainslie et al. | 2023 | **GQA** — interpolate MHA ↔ MQA | LLaMA-2/3 default | [arxiv.org/abs/2305.13245](https://arxiv.org/abs/2305.13245) |
| **FlashAttention** | Dao et al. | NeurIPS 2022 | IO-aware exact attention; less peak SRAM | [arxiv.org/abs/2205.14135](https://arxiv.org/abs/2205.14135) |
| **FlashAttention-2** | Dao | 2023 | Better parallelism for long sequences | [arxiv.org/abs/2307.08691](https://arxiv.org/abs/2307.08691) |
| **PagedAttention (vLLM)** | Kwon et al. | SOSP 2023 | **Paged KV** — OS-style virtual memory for KV | Less fragmentation | [arxiv.org/abs/2309.06180](https://arxiv.org/abs/2309.06180) |
| **H2O: Heavy-Hitter Oracle** | Zhang et al. | NeurIPS 2023 | Keep "heavy hitter" tokens; evict rest | [arxiv.org/abs/2306.14048](https://arxiv.org/abs/2306.14048) |
| **Scissorhands** | Liu et al. | ICML 2023 | Persistence-based KV eviction | [arxiv.org/abs/2305.17118](https://arxiv.org/abs/2305.17118) |
| **StreamingLLM** | Xiao et al. | 2023 | **Sliding window** + attention sink tokens | Infinite-length on fixed RAM | [arxiv.org/abs/2309.17453](https://arxiv.org/abs/2309.17453) |
| **SnapKV** | Li et al. | 2024 | Observation-window KV compression | [arxiv.org/abs/2404.14469](https://arxiv.org/abs/2404.14469) |
| **KIVI** | Liu et al. | ICML 2024 | **KV int2/int4** quantization | 2–4× KV RAM savings | [arxiv.org/abs/2402.02750](https://arxiv.org/abs/2402.02750) |
| **KVQuant** | Hooper et al. | 2024 | Per-channel KV quant + outlier handling | [arxiv.org/abs/2401.18079](https://arxiv.org/abs/2401.18079) |
| **LM-Infinite** | Han et al. | 2024 | Attention bias for infinite context | [arxiv.org/abs/2402.00788](https://arxiv.org/abs/2402.00788) |

**Story arc:** Pope et al. quantified the problem → Shazeer/Ainslie reduced KV **width** (MQA/GQA) → Dao reduced KV **peak SRAM** (FlashAttention) → Kwon reduced **fragmentation** (PagedAttention) → Zhang/Liu/Xiao reduced KV **length** (eviction/sliding) → Liu/Hooper reduced KV **precision** (quantization).

### Step 2.3 — How to IDENTIFY a KV cache problem

| # | Method | Paper / tool basis | Signal = problem |
|---|--------|-------------------|------------------|
| 1 | Formula (Pope et al.) | $\text{KV}_{\text{MB}} = 2LKh_{\text{kv}}d_h \cdot \text{elem}/10^6$ | KV > 40% RAM budget |
| 2 | OOM crash log | Android `OutOfMemoryError` during decode | Crash at token $K^\star$ |
| 3 | Latency vs $K$ | Pope et al. — attention reads full KV each step | ms/token grows with $K$ |
| 4 | Memory profiler | Android Studio RSS slope | RSS increases during generate |
| 5 | KV/budget ratio | Engineering threshold | ratio > 0.4 |
| 6 | Token cap sweep | Binary search $K$ until OOM | OOM before OCR completes |


In [ ]:
# ── Stage 2 Step 2.4: IDENTIFY KV cache ──
kv_cfg = KVCacheConfig(NUM_LAYERS, MAX_NEW_TOKENS, HIDDEN_DIM, NUM_KV_HEADS, KV_BYTES_PER_ELEM)
kv_analyzer = KVCacheAnalyzer(kv_cfg, TARGET_RAM_MB)  # Pope et al. formula + budget

rows, kv_problem = kv_analyzer.identify()  # 6 detection methods
print_detection_report("STAGE 2 — KV CACHE IDENTIFICATION (Pope et al. 2023)", rows, kv_problem, "Stage 6")

ks, mbs = kv_analyzer.growth_curve(MAX_NEW_TOKENS)  # linear growth in K
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(ks, mbs, "b-", lw=2, label="KV MB")
ax.axhline(kv_analyzer.kv_budget_mb, color="r", ls="--", label="40% RAM budget")
ax.set_xlabel("K (tokens)"); ax.set_ylabel("KV MB")
ax.set_title("Stage 2: KV grows linearly — Pope et al. 2023")
ax.legend(); plt.tight_layout(); plt.show()
print("Stage 2 done")

---
## Stage 3 — Problem 2: Power consumption

### Step 3.1 — Literature: why VLM OCR drains battery

Mobile inference energy is dominated by **DRAM traffic** (weight + KV reads) and **sustained GEMM**. Key insight from batch-serving literature:

> **Orca: A Distributed Serving System for Transformer-Based Generative Models** — Yu et al., OSDI 2022  
> Introduced **continuous batching** — batch decode steps across requests for GPU utilization.  
> [arxiv.org/abs/2206.16024](https://arxiv.org/abs/2206.16024)

On a **single phone** we cannot batch across users, but the same math applies: each decode step is a memory-bound GEMM reading full weights + growing KV.

### Step 3.2 — Key papers (power & throughput)

| Paper | Authors | Year | Core idea | Mobile relevance |
|-------|---------|------|-----------|------------------|
| **Orca** | Yu et al. | OSDI 2022 | Continuous batching for throughput | Batch PDF pages on server; phone = serial | [arxiv.org/abs/2206.16024](https://arxiv.org/abs/2206.16024) |
| **Speculative Decoding** | Leviathan et al. | ICML 2023 | Draft model + verify — fewer full forward passes | [arxiv.org/abs/2211.17192](https://arxiv.org/abs/2211.17192) |
| **Speculative Decoding (parallel)** | Chen et al. | ICML 2023 | Same idea, different formulation | [arxiv.org/abs/2302.01318](https://arxiv.org/abs/2302.01318) |
| **Medusa** | Cai et al. | 2024 | Multiple decode heads — batch token prediction | [arxiv.org/abs/2401.10774](https://arxiv.org/abs/2401.10774) |
| **Splitwise** | Patel et al. | ASPLOS 2024 | Split prompt vs token phases across chips | [arxiv.org/abs/2311.18677](https://arxiv.org/abs/2311.18677) |
| **PowerInfer** | Song et al. | 2024 | Hot-neuron locality for CPU inference | [arxiv.org/abs/2312.12456](https://arxiv.org/abs/2312.12456) |
| **LLM in a Flash** | Apple | 2024 | Flash storage for weight streaming | [arxiv.org/abs/2312.11514](https://arxiv.org/abs/2312.11514) |
| **MobileLLM** | Liu et al. | 2024 | Small models optimized for mobile | [arxiv.org/abs/2402.14905](https://arxiv.org/abs/2402.14905) |

**Story arc:** Yu et al. showed batching improves **throughput** on server → Leviathan/Chen showed **speculative decoding** cuts full-model steps (less energy per token) → Patel split phases → Song/Apple optimized for **mobile hardware constraints**.

For OCR on phone: we usually run **serial single-page decode** (no cross-user batching), so power $\propto K \times \text{FLOPs}_{\text{step}} + \text{FLOPs}_{\text{vision}}$.

### Step 3.3 — How to IDENTIFY a power problem

| # | Method | Paper basis | Signal = problem |
|---|--------|-------------|------------------|
| 1 | mAh estimate | $E = \eta \cdot \text{FLOPs}$ | > `TARGET_MAH_PER_PAGE` |
| 2 | Wall time | Orca — latency = vision + $K \times t_{\text{step}}$ | > `TARGET_LATENCY_SEC` |
| 3 | Thermal API | Android `THERMAL_STATUS_*` | MODERATE+ after 1 page |
| 4 | Battery Historian | Google discharge profiler | > 500 mA sustained |
| 5 | CPU throttle | Frequency drop mid-decode | > 30% GFLOP/s drop |
| 6 | Repeat slowdown | Thermal throttling | 2nd page > 20% slower |


In [ ]:
# ── Stage 3 Step 3.4: IDENTIFY power ──
pwr_analyzer = PowerAnalyzer()  # FLOP-based energy model (Orca + Leviathan)
rows, power_problem, est = pwr_analyzer.identify(MAX_IMAGE_SIZE, MAX_NEW_TOKENS)
print_detection_report("STAGE 3 — POWER IDENTIFICATION", rows, power_problem, "Stage 7")

flop = est.flop  # vision + decode breakdown
print(f"  FLOP breakdown: vision {flop.vision_gflops:.0f} GFLOP + decode {flop.decode_gflops:.0f} GFLOP = {flop.total_gflops:.0f} GFLOP")
print(f"  Wall time ≈ {est.seconds:.1f}s  |  mAh ≈ {est.mah:.1f}  |  thermal={est.thermal_risk}")
print("Stage 3 done")

---
## Stage 4 — Problem 3: Quantization precision loss

### Step 4.1 — Literature: post-training quantization for LLMs/VLMs

Quantization maps fp16 weights to low-bit integers. Naive round-to-nearest (RTN) fails on outliers. The field progressed rapidly 2022–2025:

### Step 4.2 — Key papers (quantization — see also notebook 02)

| Paper | Authors | Venue | Core idea | Link |
|-------|---------|-------|-----------|------|
| **LLM.int8()** | Dettmers et al. | NeurIPS 2022 | Mixed int8/fp16 with outlier columns | [arxiv.org/abs/2208.07339](https://arxiv.org/abs/2208.07339) |
| **GPTQ** | Frantar et al. | ICLR 2023 | Hessian-aware column quant | [arxiv.org/abs/2210.17323](https://arxiv.org/abs/2210.17323) |
| **AWQ** | Lin et al. | MLSys 2024 | Activation-aware scale search | [arxiv.org/abs/2306.00978](https://arxiv.org/abs/2306.00978) |
| **SmoothQuant** | Xiao et al. | ICML 2023 | Outlier migration $s_j$ | [arxiv.org/abs/2211.10438](https://arxiv.org/abs/2211.10438) |
| **SpinQuant** | Liu et al. | ICLR 2025 | Learned rotations before quant | [arxiv.org/abs/2405.16406](https://arxiv.org/abs/2405.16406) |
| **SqueezeLLM** | Kim et al. | ICML 2024 | Sensitive channel preservation | [arxiv.org/abs/2306.07682](https://arxiv.org/abs/2306.07682) |
| **QuaRot** | Ashkboos et al. | 2024 | Hadamard rotation for quant-friendly weights | [arxiv.org/abs/2404.00456](https://arxiv.org/abs/2404.00456) |
| **OmniQuant** | Shao et al. | ICLR 2024 | Learnable clipping + equivalent transform | [arxiv.org/abs/2308.13137](https://arxiv.org/abs/2308.13137) |
| **QServe** | Lin et al. | 2024 | W4A8KV4 serving | [arxiv.org/abs/2405.04532](https://arxiv.org/abs/2405.04532) |

**Story arc:** Dettmers found **outlier columns** break int8 → Frantar used **Hessian** (GPTQ) → Lin searched **activation scales** (AWQ) → Xiao **migrated outliers** (SmoothQuant) → Kim preserved **sensitive channels** (SqueezeLLM) → rotation methods (SpinQuant, QuaRot) spread outliers before quant.

For OCR: errors in `lm_head` → garbled tokens; errors in vision projection → **box drift**.

### Step 4.3 — How to IDENTIFY quant precision loss

| # | Method | Paper basis | Signal = problem |
|---|--------|-------------|------------------|
| 1 | Weight MSE | GPTQ objective proxy | top-layer MSE > $10^{-3}$ |
| 2 | Output MSE | AWQ calibration metric | output MSE > $10^{-2}$ |
| 3 | Line recall | OCR task metric | < `MIN_OCR_LINE_RECALL` |
| 4 | CER | Speech/OCR standard | CER > 5% vs fp16 |
| 5 | Box IoU | Detection metric | mean IoU < 0.85 |
| 6 | Visual inspection | Human eval | any regression |


In [ ]:
# ── Stage 4 Step 4.4: IDENTIFY quant loss ──
from transformers import AutoProcessor, AutoModelForCausalLM
from PIL import Image, ImageDraw
from io import BytesIO
import requests

dtype = torch.float16 if DEVICE == "cuda" else torch.float32
processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(  # load Florence-2 for layer profiling
    MODEL_ID, trust_remote_code=True, torch_dtype=dtype, attn_implementation="eager"
).to(DEVICE).eval()

def load_img():
    try:
        u = "https://raw.githubusercontent.com/Gaurav14cs17/Document-OCR-Pipeline/main/assets/table_page.png"
        return Image.open(BytesIO(requests.get(u, timeout=30).content)).convert("RGB")
    except Exception:
        im = Image.new("RGB", (640, 480), "white")
        ImageDraw.Draw(im).text((20, 20), "Sample", fill="black")
        return im

quant_analyzer = QuantAnalyzer(model, bits=QUANT_BITS)  # GPTQ/AWQ proxy via RTN MSE
layer_profiles = quant_analyzer.profile_layers(max_layers=40)  # SqueezeLLM sensitivity ranking
rows, quant_problem = quant_analyzer.identify(layer_profiles, recall=1.0)
print_detection_report("STAGE 4 — QUANT IDENTIFICATION (GPTQ/AWQ metrics)", rows, quant_problem, "Stage 8")

print("  Top sensitive layers (preserve in mixed precision):")
for lp in layer_profiles[:5]:
    tag = " [PROTECT]" if lp.protected else ""
    print(f"    {lp.sensitivity:.2e}  {lp.name.split('.')[-1]}{tag}")
print("Stage 4 done")

---
## Stage 5 — Problem 4: High RAM consumption

### Step 5.1 — Literature: memory wall for inference

Peak RAM is not disk size. Key analyses:

> **Efficiently Scaling Transformer Inference** — Pope et al., MLSys 2023  
> Peak memory = weights + activations + KV; batching multiplies KV.  
> [arxiv.org/abs/2207.04910](https://arxiv.org/abs/2207.04910)

> **FlexGen: High-Throughput Generative Inference of LLMs with a Single GPU** — Sheng et al., ICML 2023  
> **Offloading** weights/KV to CPU/disk when GPU RAM insufficient.  
> [arxiv.org/abs/2303.06865](https://arxiv.org/abs/2303.06865)

### Step 5.2 — Key papers (memory-efficient inference)

| Paper | Authors | Year | Core idea | Mobile relevance |
|-------|---------|------|-----------|------------------|
| **FlexGen** | Sheng et al. | ICML 2023 | GPU-CPU-disk tiered memory | Phone = mmap flash | [arxiv.org/abs/2303.06865](https://arxiv.org/abs/2303.06865) |
| **LLM in a Flash** | Apple | 2024 | Flash-backed weight streaming | Direct mobile analog | [arxiv.org/abs/2312.11514](https://arxiv.org/abs/2312.11514) |
| **ZeRO-Inference** | Aminabadi et al. | 2022 | Partition model across devices | [arxiv.org/abs/2207.07660](https://arxiv.org/abs/2207.07660) |
| **ZeroQuant** | Yao et al. | 2022 | INT8 quant + KV cache quant | [arxiv.org/abs/2206.01861](https://arxiv.org/abs/2206.01861) |
| **DeepSpeed-Inference** | Aminabadi et al. | 2022 | Multi-GPU inference engine | Server-side |
| **llama.cpp** | Gerganov | 2023 | mmap + quant on CPU/mobile | Used in many mobile apps | [github.com/ggerganov/llama.cpp](https://github.com/ggerganov/llama.cpp) |

**Story arc:** Pope quantified components → Sheng **offloaded** to cheaper memory → Apple **streamed from flash** → Gerganov shipped **mmap + int4** on phone → our notebook 04 implements the same pattern.

$$\text{RAM}_{\text{peak}} = W + I + A_v + \text{KV}(K) + A_{\text{dec}}$$

### Step 5.3 — How to IDENTIFY RAM problem

| # | Method | Paper basis | Signal = problem |
|---|--------|-------------|------------------|
| 1 | Peak formula | Pope et al. 2023 | > `TARGET_RAM_MB` |
| 2 | Dominant term | Engineering analysis | KV or W > 50% |
| 3 | Memory profiler | Android Studio | RSS > budget |
| 4 | Low-memory kill | `onTrimMemory` | app killed |
| 5 | fp16 vs int4 gap | Quant impact on W | fp16 alone OOM |
| 6 | Camera overlap | FlexGen tiering insight | camera + model > budget |


In [ ]:
# ── Stage 5 Step 5.4: IDENTIFY RAM ──
params = int(PARAMS_M * 1e6)  # use config estimate (or model.parameters() after Stage 4)
w_fp16 = params * 2 / (1024 ** 2)  # fp16 weight footprint
w_int4 = params * 0.5 / (1024 ** 2)  # int4 ≈ 0.5 B/param

ram_analyzer = RAMAnalyzer()  # Pope et al. component model
ram_fp16 = ram_analyzer.estimate(w_fp16, MAX_IMAGE_SIZE, kv_cfg)
ram_int4 = ram_analyzer.estimate(w_int4, MAX_IMAGE_SIZE, kv_cfg)
rows, ram_problem = ram_analyzer.identify(ram_int4.peak_mb, ram_int4, ram_fp16.peak_mb)
print_detection_report("STAGE 5 — RAM IDENTIFICATION (Pope et al. / FlexGen)", rows, ram_problem, "Stage 9")

print("\nPART A SUMMARY:")
print(f"  KV={kv_problem}  Power={power_problem}  Quant={quant_problem}  RAM={ram_problem}")
print("Stage 5 done — proceed to PART B")

---
# PART B — SOLVE

---
## Stage 6 — Fix KV cache (paper-backed methods)

### Step 6.1 — All fix methods with paper references

| # | Fix | Paper | How | Savings | Priority |
|---|-----|-------|-----|---------|----------|
| 1 | Cap `max_new_tokens` | Pope et al. 2023 | Limit $K$ | linear in $K$ | **P0** |
| 2 | **GQA** | Ainslie et al. 2023 | $h_{\text{kv}} < h$ | $h/h_{\text{kv}}$ | **P0** |
| 3 | **MQA** | Shazeer 2019 | $h_{\text{kv}}=1$ | $N_h \times$ | **P1** |
| 4 | **KV int8/int4** | KIVI 2024, KVQuant 2024 | Quantize KV tensors | 2–4× | **P1** |
| 5 | **Sliding window** | StreamingLLM 2023 | Keep last $W$ tokens + sink | bounded | **P1** |
| 6 | **KV eviction** | H2O 2023, Scissorhands 2023, SnapKV 2024 | Drop low-importance tokens | sub-linear | **P2** |
| 7 | **PagedAttention** | Kwon et al. 2023 (vLLM) | Non-contiguous KV pages | less fragmentation | **P2** |
| 8 | **FlashAttention** | Dao 2022/2023 | IO-aware attention | lower peak SRAM | **P2** |
| 9 | `use_cache=False` | — | Recompute (slow) | 100% KV RAM | **P3** |


In [ ]:
# ── Stage 6 Step 6.2: SOLVE KV cache ──

class KVCacheFixSimulator:
    """Paper-backed KV mitigations — each method returns a FixResult."""

    def __init__(self, cfg: KVCacheConfig):
        self.cfg = cfg
        self.baseline_mb = cfg.mb

    def _fix(self, name: str, paper: str, priority: str, new_cfg: KVCacheConfig) -> FixResult:
        val = new_cfg.mb
        saved = 1.0 - val / max(self.baseline_mb, 1e-6)
        return FixResult(name, paper, priority, val, "MB", saved)

    def apply_gqa(self, kv_heads: int = 4) -> FixResult:
        return self._fix("GQA", "Ainslie et al. 2023", "P0",
                         self.cfg.with_overrides(num_kv_heads=kv_heads))

    def apply_mqa(self) -> FixResult:
        return self._fix("MQA", "Shazeer 2019", "P1",
                         self.cfg.with_overrides(num_kv_heads=1))

    def apply_kivi_int8(self) -> FixResult:
        return self._fix("KIVI int8", "Liu et al. ICML 2024", "P1",
                         self.cfg.with_overrides(bytes_per_elem=1))

    def apply_streaming_llm(self, window: int = 64) -> FixResult:
        return self._fix("StreamingLLM", "Xiao et al. 2023", "P1",
                         self.cfg.with_overrides(seq_len=min(window, self.cfg.seq_len)))

    def apply_h2o(self, keep_ratio: float = 0.5) -> FixResult:
        kept = max(1, int(self.cfg.seq_len * keep_ratio))
        return self._fix("H2O eviction", "Zhang et al. NeurIPS 2023", "P2",
                         self.cfg.with_overrides(seq_len=kept))

    def apply_no_cache(self) -> FixResult:
        return self._fix("use_cache=False", "recompute (slow)", "P3",
                         self.cfg.with_overrides(seq_len=0))

    def all_fixes(self) -> list[FixResult]:
        return [
            FixResult("Cap K=128", "Pope et al. 2023", "P0",
                      KVCacheConfig(self.cfg.num_layers, min(128, self.cfg.seq_len),
                                    self.cfg.hidden_dim, self.cfg.num_kv_heads, 2).mb, "MB",
                      1 - min(128, self.cfg.seq_len) / max(self.cfg.seq_len, 1)),
            self.apply_gqa(4),
            self.apply_mqa(),
            self.apply_kivi_int8(),
            self.apply_streaming_llm(64),
            self.apply_h2o(0.5),
            self.apply_no_cache(),
        ]


kv_sim = KVCacheFixSimulator(kv_cfg)
kv_fixes = kv_sim.all_fixes()
print_fix_report("STAGE 6 — KV FIXES (paper-backed)", kv_sim.baseline_mb, kv_fixes, "MB")

rec_kv = {"max_new_tokens": 128, "kv_num_heads": 4, "kv_bytes_per_elem": 1,
          "sliding_window": 64, "method": "GQA+KIVI+StreamingLLM"}
print(f"Recommended: {rec_kv}")
print("Stage 6 done")

---
## Stage 7 — Fix power consumption (paper-backed methods)

### Step 7.1 — All fix methods with paper references

| # | Fix | Paper | How | Impact | Priority |
|---|-----|-------|-----|--------|----------|
| 1 | Cap $K$ | Pope/Orca | Fewer decode steps | ↓ decode energy | **P0** |
| 2 | Downscale image | Pope et al. | $\text{FLOPs}_v \propto s^2$ | ↓ vision energy | **P0** |
| 3 | **Speculative decoding** | Leviathan 2023, Chen 2023 | Draft + verify | fewer full steps | **P1** |
| 4 | **Medusa heads** | Cai 2024 | Batch-predict multiple tokens | ↑ tokens/step | **P2** |
| 5 | **Continuous batching** | Orca 2022 | Batch PDF pages on server | ↑ throughput/W | **P2** (server) |
| 6 | int4 weights | GPTQ/AWQ | Less DRAM traffic | ↓ memory energy | **P1** |
| 7 | NPU delegate | MobileLLM 2024 | Hardware accelerator | 3–5× J/GFLOP | **P1** |
| 8 | Throttle + charger batch | Engineering | Spread heat over time | ↓ thermal | **P1** |

**Batch decoding story (Orca):** On server, **continuous batching** merges decode steps from multiple requests into one GPU kernel → higher utilization, lower energy **per token**. On phone for PDF OCR: batch pages **on server**; on device process **serially with throttle**.

**Speculative decoding story:** Small draft model proposes $\gamma$ tokens; large model verifies in **one parallel forward** → accept/reject. Average accepted tokens per full step $> 1$ → less energy per output token (Leviathan et al., ICML 2023).


In [ ]:
# ── Stage 7 Step 7.2: SOLVE power ──

class PowerFixSimulator:
    """Power mitigations — speculative decoding math + Orca batch note."""

    def __init__(self, analyzer: PowerAnalyzer, image_side: int, num_tokens: int):
        self.analyzer = analyzer
        self.image_side = image_side
        self.num_tokens = num_tokens
        self.baseline = analyzer.estimate_power(image_side, num_tokens, quant=False)

    def speculative_speedup(self, gamma: int = 4, accept_rate: float = 0.7) -> float:
        # Leviathan 2023: accepted tokens per full-model step ≈ 1 + γ·α
        return 1.0 + gamma * accept_rate

    def apply_fix(self, name: str, paper: str, priority: str, **kwargs) -> FixResult:
        side = kwargs.pop("image_side", self.image_side)
        tokens = kwargs.pop("num_tokens", self.num_tokens)
        est = self.analyzer.estimate_power(side, tokens, **kwargs)
        saved = 1.0 - est.mah / max(self.baseline.mah, 1e-6)
        return FixResult(name, paper, priority, est.mah, "mAh", saved)

    def all_fixes(self) -> list[FixResult]:
        fixes = [
            self.apply_fix("Cap K=128", "Pope/Orca", "P0", num_tokens=min(128, self.num_tokens)),
            self.apply_fix("Image 512px", "Pope et al. 2023", "P0", image_side=512),
            self.apply_fix("Speculative dec.", "Leviathan ICML 2023", "P1", speculative=True),
            self.apply_fix("int4 (GPTQ/AWQ)", "Frantar/Lin", "P1", quant=True),
            FixResult("NPU 8× faster", "MobileLLM 2024", "P1",
                      self.baseline.mah * 0.5, "mAh", 0.50),
            FixResult("Orca batch (server)", "Yu OSDI 2022", "P2",
                      self.baseline.mah, "mAh", 0.0),
        ]
        return fixes


pwr_sim = PowerFixSimulator(pwr_analyzer, MAX_IMAGE_SIZE, MAX_NEW_TOKENS)
pwr_fixes = pwr_sim.all_fixes()
print_fix_report("STAGE 7 — POWER FIXES", pwr_sim.baseline.mah, pwr_fixes, "mAh")

spd = pwr_sim.speculative_speedup(gamma=4, accept_rate=0.7)
print(f"  Speculative decoding: γ=4, α=0.7 → {spd:.1f}× tokens per full step (Leviathan 2023)")
print("  Orca note: continuous batching helps server throughput; phone OCR = serial pages + throttle")
rec_pwr = {"max_new_tokens": 128, "max_image_size": 512, "speculative": True,
           "throttle_ms": 3000, "batch_pdf_on_server": True}
print(f"\nRecommended: {rec_pwr}")
print("Stage 7 done")

---
## Stage 8 — Fix quant precision loss (paper-backed methods)

### Step 8.1 — All fix methods with paper references

| # | Fix | Paper | How | Priority |
|---|-----|-------|-----|----------|
| 1 | fp16 `lm_head` | SqueezeLLM 2024 | Preserve sensitive output channels | **P0** |
| 2 | fp16 embeddings | LLM.int8() 2022 | Outlier-safe lookup | **P0** |
| 3 | **GPTQ** | Frantar 2023 | Hessian-aware column quant | **P1** |
| 4 | **AWQ** | Lin 2024 | Activation-aware scales | **P1** |
| 5 | **SmoothQuant** | Xiao 2023 | Outlier migration | **P1** |
| 6 | **SpinQuant** | Liu 2025 | Learned rotations | **P2** |
| 7 | **Mixed precision** | LLM.int8() + nb02 | fp16/int8/int4 per layer | **P1** |
| 8 | fp16 vision proj | AWQ sensitivity | Stable bounding boxes | **P1** |
| 9 | int8 mid-tier | OmniQuant 2024 | Safer than int4 on borderline | **P2** |

Full implementations in [notebook 02](02_ocr_pipeline_quant.ipynb).


In [ ]:
# ── Stage 8 Step 8.2: SOLVE quant ──

class QuantFixPlanner:
    """Mixed-precision assignment — SqueezeLLM + LLM.int8() patterns."""

    def __init__(self, layers: list[LayerQuantProfile], bits: int = QUANT_BITS):
        self.layers = layers
        self.bits = bits

    def mixed_precision_assignment(self) -> dict[str, str]:
        n_fp16 = max(1, int(len(self.layers) * FP16_SENSITIVE_PCT / 100))
        assignment: dict[str, str] = {}
        for i, lp in enumerate(self.layers):
            if lp.protected or any(p in lp.name.lower() for p in ALWAYS_FP16_PATTERNS):
                assignment[lp.name] = "fp16"  # always protect vision + lm_head
            elif i < n_fp16:
                assignment[lp.name] = "fp16"  # top sensitive layers → fp16
            elif lp.sensitivity > 1e-4:
                assignment[lp.name] = "int8"  # borderline → int8 (OmniQuant tier)
            else:
                assignment[lp.name] = f"int{self.bits}"  # bulk → int4
        return assignment

    def summarize(self) -> dict[str, int]:
        a = self.mixed_precision_assignment()
        counts: dict[str, int] = {}
        for v in a.values():
            counts[v] = counts.get(v, 0) + 1
        return counts

    def fix_catalog(self) -> list[tuple[str, str, str]]:
        return [
            ("P0 fp16 lm_head", "SqueezeLLM 2024", "preserve output channels"),
            ("P0 fp16 embed", "LLM.int8() 2022", "outlier-safe lookup"),
            ("P1 GPTQ", "Frantar 2023", "Hessian column quant"),
            ("P1 AWQ", "Lin 2024", "activation scale search"),
            ("P1 SmoothQuant", "Xiao 2023", "outlier migration"),
            ("P1 Mixed precision", f"top {FP16_SENSITIVE_PCT}% fp16 + int8 + int4", "per-layer"),
            ("P2 SpinQuant", "Liu 2025", "learned Givens rotations"),
            ("P2 int8 borderline", "OmniQuant 2024", "safer mid-tier"),
        ]


quant_planner = QuantFixPlanner(layer_profiles)
counts = quant_planner.summarize()
print("STAGE 8 — QUANT FIXES\n")
for name, paper, how in quant_planner.fix_catalog():
    print(f"  {name:<22} {paper:<20} {how}")
print(f"\n  Mixed precision: {counts}")
rec_quant = {"method": "awq", "bits": QUANT_BITS, "fp16_sensitive_pct": FP16_SENSITIVE_PCT,
             "always_fp16": list(ALWAYS_FP16_PATTERNS), "assignment_sample": dict(list(quant_planner.mixed_precision_assignment().items())[:5])}
print(f"\nRecommended: {rec_quant}")
print("Stage 8 done")

---
## Stage 9 — Fix RAM (paper-backed methods)

### Step 9.1 — All fix methods with paper references

| # | Fix | Paper | How | Priority |
|---|-----|-------|-----|----------|
| 1 | **int4 weights** | GPTQ/AWQ | 4× smaller $W$ | **P0** |
| 2 | **mmap / flash stream** | llama.cpp, Apple 2024 | Page weights from flash | **P0** |
| 3 | **FlexGen offloading** | Sheng 2023 | Tier GPU/CPU/disk | **P1** (server) |
| 4 | Split vision/lang load | Pope 2023 | $\max(|\theta_v|, |\theta_l|+\text{KV})$ | **P1** |
| 5 | Release camera buffer | Engineering | Free $I_{\text{bmp}}$ | **P1** |
| 6 | KV fixes (Stage 6) | KIVI/StreamingLLM | Smaller KV term | **P1** |
| 7 | Downscale image | Pope 2023 | $I \propto s^2$ | **P2** |
| 8 | ZeroQuant KV int8 | Yao 2022 | Quantize KV + weights | **P2** |


In [ ]:
# ── Stage 9 Step 9.2: SOLVE RAM ──

class RAMFixSimulator:
    """Combine weight/KV/image fixes — Pope + FlexGen + llama.cpp patterns."""

    def __init__(self, analyzer: RAMAnalyzer, w_fp16: float, w_int4: float, kv_cfg: KVCacheConfig):
        self.analyzer = analyzer
        self.w_fp16 = w_fp16
        self.w_int4 = w_int4
        self.kv_cfg = kv_cfg
        self.baseline = analyzer.estimate(w_fp16, MAX_IMAGE_SIZE, kv_cfg)

    def _fix(self, name: str, paper: str, priority: str, breakdown: RAMBreakdown) -> FixResult:
        saved = 1.0 - breakdown.peak_mb / max(self.baseline.peak_mb, 1e-6)
        return FixResult(name, paper, priority, breakdown.peak_mb, "MB", saved)

    def all_fixes(self) -> list[FixResult]:
        fixes = [
            self._fix("int4 (GPTQ/AWQ)", "Frantar/Lin", "P0",
                      self.analyzer.estimate(self.w_int4, MAX_IMAGE_SIZE, self.kv_cfg)),
            self._fix("mmap (llama.cpp)", "Gerganov 2023", "P0",
                      self.analyzer.estimate(self.w_int4 * 0.5, MAX_IMAGE_SIZE, self.kv_cfg)),
            self._fix("split vision/lang", "Pope 2023", "P1",
                      self.analyzer.estimate(self.w_int4 * 0.7, MAX_IMAGE_SIZE, self.kv_cfg)),
        ]
        base_int4 = self.analyzer.estimate(self.w_int4, MAX_IMAGE_SIZE, self.kv_cfg)
        no_cam = RAMBreakdown(base_int4.weights_mb, 0, base_int4.image_tensor_mb,
                              base_int4.vision_activation_mb, base_int4.kv_cache_mb,
                              base_int4.decode_activation_mb)
        fixes.append(self._fix("release camera", "engineering", "P1", no_cam))
        kv_fixed = KVCacheConfig(NUM_LAYERS, 128, HIDDEN_DIM, 4, 1)
        fixes.append(self._fix("KV fixes Stg6", "KIVI+GQA", "P1",
                               self.analyzer.estimate(self.w_int4, MAX_IMAGE_SIZE, kv_fixed)))
        fixes.append(self._fix("downscale 512", "Pope 2023", "P2",
                               self.analyzer.estimate(self.w_int4, 512, kv_fixed)))
        return fixes


ram_sim = RAMFixSimulator(ram_analyzer, w_fp16, w_int4, kv_cfg)
ram_fixes = ram_sim.all_fixes()
print_fix_report("STAGE 9 — RAM FIXES", ram_sim.baseline.peak_mb, ram_fixes, "MB")

rec_ram = {"mmap": True, "split_load": True, "release_camera": True, **rec_kv, "max_image_size": 512}
print(f"Recommended: {rec_ram}")
print("Stage 9 done")

---
# PART C — VERIFY

## Stage 10 — Scorecard + paper reference index

### Full paper bibliography (Notebook 05)

| Topic | Papers cited |
|-------|-------------|
| **KV cache** | Vaswani 2017, Shazeer 2019, Ainslie 2023, Dao 2022/23, Kwon 2023, Zhang 2023, Liu 2023, Xiao 2023, Li 2024, Liu 2024 (KIVI), Hooper 2024, Han 2024 |
| **Power** | Yu 2022 (Orca), Leviathan 2023, Chen 2023, Cai 2024 (Medusa), Patel 2024, Song 2024, Apple 2024, Liu 2024 (MobileLLM) |
| **Quant** | Dettmers 2022, Frantar 2023, Lin 2024, Xiao 2023, Liu 2025, Kim 2024, Ashkboos 2024, Shao 2024, Lin 2024 (QServe) |
| **RAM** | Pope 2023, Sheng 2023 (FlexGen), Aminabadi 2022, Yao 2022, Apple 2024, llama.cpp |


In [ ]:
# ── Stage 10: Before/after scorecard + export ──
# ProductionScorecard defined in Stage 1

# Fixed config after Part B mitigations
fixed_kv_cfg = KVCacheConfig(NUM_LAYERS, 128, HIDDEN_DIM, 4, 1)
fixed_ram = ram_analyzer.estimate(w_int4 * 0.5, 512, fixed_kv_cfg)
fixed_pwr = pwr_analyzer.estimate_power(512, 128, quant=True, speculative=True)

before = ProductionScorecard(
    kv_mb=kv_cfg.mb, kv_budget=TARGET_RAM_MB * 0.4,
    mah=pwr_sim.baseline.mah, mah_budget=TARGET_MAH_PER_PAGE,
    ram_mb=ram_fp16.peak_mb, ram_budget=TARGET_RAM_MB,
    recall=0.80 if quant_problem else 1.0, recall_min=MIN_OCR_LINE_RECALL,
    latency_s=pwr_sim.baseline.seconds, latency_budget=TARGET_LATENCY_SEC,
    label="BEFORE (Part A baseline)",
)
after = ProductionScorecard(
    kv_mb=fixed_kv_cfg.mb, kv_budget=TARGET_RAM_MB * 0.4,
    mah=fixed_pwr.mah, mah_budget=TARGET_MAH_PER_PAGE,
    ram_mb=fixed_ram.peak_mb, ram_budget=TARGET_RAM_MB,
    recall=0.92, recall_min=MIN_OCR_LINE_RECALL,
    latency_s=fixed_pwr.seconds, latency_budget=TARGET_LATENCY_SEC,
    label="AFTER (Part B fixes applied)",
)
before.print_card()
after.print_card()

config = {
    **rec_kv, **rec_pwr, **rec_quant, **rec_ram,
    "target_ram_mb": TARGET_RAM_MB,
    "target_mah_per_page": TARGET_MAH_PER_PAGE,
    "scorecard_before": {r[0]: r[1] for r in before.rows()},
    "scorecard_after": {r[0]: r[1] for r in after.rows()},
    "papers": {
        "kv": ["Vaswani2017", "Shazeer2019", "Ainslie2023-GQA", "Dao2022-FlashAttn",
               "Kwon2023-PagedAttn", "Zhang2023-H2O", "Xiao2023-StreamingLLM", "Liu2024-KIVI"],
        "power": ["Yu2022-Orca", "Leviathan2023-SpecDec", "Chen2023-SpecDec", "Cai2024-Medusa"],
        "quant": ["Dettmers2022-LLMint8", "Frantar2023-GPTQ", "Lin2024-AWQ", "Xiao2023-SmoothQuant"],
        "ram": ["Pope2023", "Sheng2023-FlexGen", "Apple2024-LLMFlash", "llama.cpp"],
    },
}
out_path = "mobile_production_mitigations.json"
with open(out_path, "w") as f:
    json.dump(config, f, indent=2)
print(f"\nExported {out_path}")
print("Stage 10 done — notebook complete")

---
## Summary — step-by-step workflow

| Step | Action | Papers to read |
|------|--------|----------------|
| 1 | Run Stage 0–1 (config + utils) | — |
| 2 | **Identify KV** (Stage 2) | Pope 2023, Ainslie 2023 (GQA), Kwon 2023 |
| 3 | **Identify Power** (Stage 3) | Yu 2022 (Orca), Leviathan 2023 |
| 4 | **Identify Quant** (Stage 4) | Frantar 2023, Lin 2024, Dettmers 2022 |
| 5 | **Identify RAM** (Stage 5) | Pope 2023, Sheng 2023 (FlexGen) |
| 6 | **Fix KV** (Stage 6) | GQA, KIVI, StreamingLLM |
| 7 | **Fix Power** (Stage 7) | Speculative decoding, Orca batching |
| 8 | **Fix Quant** (Stage 8) | GPTQ/AWQ/SmoothQuant (notebook 02) |
| 9 | **Fix RAM** (Stage 9) | mmap, llama.cpp, Apple flash streaming |
| 10 | **Verify** (Stage 10) | Re-run scorecard until all green |

**Series:** [01 OCR](01_document_ocr_pipeline.ipynb) → [02 Quant](02_ocr_pipeline_quant.ipynb) → [03 Mobile](03_ocr_pipeline_mobile.ipynb) → [04 Complete](04_ocr_pipeline_mobile_complete.ipynb) → **05 Issues**
